# 📊 08｜外送訂單：先把 UTC 換成台灣時間，再看尖峰時段

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/08_delivery_time/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：時間欄位是 **UTC（世界標準時間）**，比台灣慢 8 小時；直接取「小時」會把午餐尖峰算成凌晨 4 點。要先**轉成台灣時間**，再拆出小時、日期。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：發現時間是 UTC、轉成台灣時間，最後用**一張垂直長條圖**回答「一天中哪個時段訂單最多？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出各小時訂單數與占比，作為排班參考。
3. **選做挑戰（單元五～六）**：比較各區送達時間，或觀察每日訂單走勢與特殊節日。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/delivery_orders.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一筆外送訂單**，共 2,500 筆，期間為台灣時間 2025-05-01～06-30。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`訂單編號` ➔ 只能計數。
- 🔵 **條件 (Dimension)**：`下單時間_utc`、`送達時間_utc`（時間，**UTC 時區**）；`區域`（類別）。
- 🟢 **指標 (Metric)**：`訂單金額`（元）；衍生指標 `送達分鐘`＝送達時間 − 下單時間。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：以台灣時間來看，一天中哪個小時的訂單最多？
- 🛵 **比較＋分布（選做）**：各區域的送達時間有差別嗎？
- 📈 **趨勢（選做）**：兩個月的每日訂單量怎麼變化？有沒有特別突出的日子？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **先確認時區**：時間結尾的 Z 代表 UTC；不轉換就取小時，午餐尖峰會出現在凌晨 3～4 點。
- ⚠️ **日期也會跟著變**：UTC 16:00 之後，在台灣已經是隔天；依日期統計時也要用台灣時間。
- ⚠️ **送達時間不等於外送員效率**：資料沒有距離、天氣、餐廳出餐時間，不能說某區外送員比較慢。
- ⚠️ **特殊日子會拉高走勢**：5/31 端午節訂單特別多，看趨勢時要標出來，不能直接當成成長。

### 4. 決定最適圖表
- **24 個小時比訂單量** ➔ **垂直長條圖**（小時是有順序的類別，0→23 排列，不可依數量重排；Y 軸從 0 開始）。
- **各區送達分鐘（選做）** ➔ **箱形圖**（多組比較分布）。
- **每日訂單走勢（選做）** ➔ **折線圖**，加上 7 日移動平均線，標出端午節。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'delivery_orders.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/08_delivery_time/data/delivery_orders.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料，辨認時間欄位的時區（貼標籤）

> 🎯 **思維重點**：時間欄位看起來像「2025-05-01T03:15:22Z」：T 分隔日期與時間，**Z 代表 UTC**。先看懂格式，再決定怎麼轉換。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/delivery_orders.csv：
1. 顯示前 5 筆資料、所有欄位名稱、總列數、資料型態與缺值數。
2. 說明「下單時間_utc」的格式：T 和結尾的 Z 分別代表什麼？台灣時間和 UTC 差幾小時？
3. 用文字說明：一列代表什麼？哪些欄位是標識、條件、指標？
先不要轉換時間，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/delivery_orders.csv')
print("--- 前 5 筆資料 ---")
display(df.head())
print(f"\n總列數：{len(df)}；缺值：{df.isna().sum().to_dict()}")
df.info()
print("\n時間格式範例：", df.loc[0, '下單時間_utc'])
print("➡️ T 分隔日期與時間；結尾的 Z 代表 UTC（世界標準時間），台灣時間 = UTC + 8 小時。")

### 📝 單元二【必做】：邊界檢核：UTC 轉台灣時間，比較轉換前後的尖峰

> 🎯 **思維重點**：先示範「沒轉時區」的錯誤尖峰，再轉成 Asia/Taipei；轉換後拆出小時與日期，並清點日期改變的筆數。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請處理時間欄位的時區：
1. 先示範錯誤做法：直接從「下單時間_utc」取出小時，列出訂單最多的前 3 個小時。
2. 正確做法：把下單時間與送達時間解析為 UTC，再轉換成台灣時區（Asia/Taipei），新增「下單時間_台灣」「送達時間_台灣」。
3. 從台灣時間拆出「小時」與「日期」，列出訂單最多的前 3 個小時。
4. 計算有幾筆訂單的日期因為時區轉換而改變（UTC 日期 ≠ 台灣日期），並顯示 3 筆範例。
5. 計算「送達分鐘」＝送達時間 − 下單時間，檢查最小值與最大值是否合理。
不要修改原始 CSV 檔。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# ❌ 錯誤示範：沒轉時區就取小時
wrong_hour = pd.to_datetime(df['下單時間_utc']).dt.hour
print("❌ 直接用 UTC 取小時，訂單最多的前 3 個小時：", wrong_hour.value_counts().head(3).to_dict())

# ✅ 正確做法：解析為 UTC，再轉成台灣時間
df['下單時間_台灣'] = pd.to_datetime(df['下單時間_utc'], utc=True).dt.tz_convert('Asia/Taipei')
df['送達時間_台灣'] = pd.to_datetime(df['送達時間_utc'], utc=True).dt.tz_convert('Asia/Taipei')
df['小時'] = df['下單時間_台灣'].dt.hour
df['日期'] = df['下單時間_台灣'].dt.date
print("✅ 台灣時間，訂單最多的前 3 個小時：", df['小時'].value_counts().head(3).to_dict())

changed = df[df['下單時間_utc'].str[:10] != df['日期'].astype(str)]
print(f"\n日期因轉換而改變：{len(changed)} 筆，例如：")
display(changed[['訂單編號', '下單時間_utc', '下單時間_台灣']].head(3))

df['送達分鐘'] = (df['送達時間_台灣'] - df['下單時間_台灣']).dt.total_seconds() / 60
print(f"送達分鐘：最短 {df['送達分鐘'].min():.0f}、最長 {df['送達分鐘'].max():.0f}、中位數 {df['送達分鐘'].median():.0f} 分鐘")

### 📝 單元三【必做】：提問積木【比較】— 一天中哪個時段訂單最多？（垂直長條圖）

> 🎯 **思維重點**：24 個小時是**有順序的類別**，用【垂直長條圖】依 0→23 排列（不能依數量重排）；Y 軸從 0 開始，標題註明「台灣時間」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用台灣時間的「小時」畫出訂單數的垂直長條圖：
1. 先產出表格：0～23 每個小時的訂單數（沒有訂單的小時也要顯示 0）。
2. X 軸為小時（0～23，依時間順序），Y 軸為訂單數，從 0 開始。
3. 訂單數最多的兩個小時用醒目顏色標示，並在柱子上標註數字。
4. 標題註明「台灣時間」，圖表文字全部使用繁體中文。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

hourly = df['小時'].value_counts().reindex(range(24), fill_value=0).rename('訂單數')
with pd.option_context('display.max_columns', 30):
    display(hourly.to_frame().T)
top2 = hourly.nlargest(2).index
colors = ['#e74c3c' if h in top2 else '#3498db' for h in hourly.index]

plt.figure(figsize=(11, 5))
plt.bar(hourly.index, hourly.values, color=colors)
for h in top2:
    plt.text(h, hourly[h] + 5, f'{hourly[h]}', ha='center', fontweight='bold')
plt.xticks(range(24))
plt.ylim(0, hourly.max() * 1.15)
plt.title('各小時外送訂單數（台灣時間，2025/5/1～6/30）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('下單時間（點，台灣時間）')
plt.ylabel('訂單數（筆）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能指出時間欄位的 Z 代表 UTC，並說明台灣時間要加 8 小時。
- [ ] 能展示轉換前後的尖峰小時差異，以及有多少筆訂單的「日期」因轉換而改變。
- [ ] 能從長條圖與表格指出台灣時間的兩個尖峰時段，並核對數字一致。
- 💬 **討論題**：如果交出「凌晨 4 點最忙」的報告，主管會做出什麼錯誤決定？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 直接用 UTC 取小時，尖峰在 **10 點與凌晨 4 點**（荒謬）；轉成台灣時間後，尖峰在 **12 點（320 筆）與 18 點（323 筆）**。
- 有 **130 筆**訂單的日期在轉換後變成隔天。
- 長條圖 X 軸依 0～23 點排列、Y 軸從 0 開始，午餐與晚餐兩個尖峰清楚可見。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出各小時訂單數與占比

> 🎯 **思維重點**：排班報表要寫清楚時區；Excel 不支援帶時區的時間，匯出明細前要先轉成一般文字或去除時區資訊。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將時段分析結果匯出成 Excel：
1. 工作表「各小時訂單」：小時（台灣時間）、訂單數、占全部訂單百分比（保留一位小數）。
2. 工作表「訂單明細」：訂單編號、下單時間_台灣、送達時間_台灣、區域、訂單金額、送達分鐘；時間轉成「YYYY-MM-DD HH:MM:SS」文字，避免 Excel 時區問題。
3. 存為 output/外送時段報表.xlsx（若 output 目錄不存在請自動建立），重新讀取核對訂單合計為 2,500。
4. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
hour_table = hourly.to_frame()
hour_table['占比(%)'] = (hour_table['訂單數'] / hour_table['訂單數'].sum() * 100).round(1)
hour_table.index.name = '小時（台灣時間）'

detail = df[['訂單編號', '下單時間_台灣', '送達時間_台灣', '區域', '訂單金額', '送達分鐘']].copy()
for col in ['下單時間_台灣', '送達時間_台灣']:
    detail[col] = detail[col].dt.strftime('%Y-%m-%d %H:%M:%S')   # Excel 不支援帶時區的時間

os.makedirs('output', exist_ok=True)
output_path = 'output/外送時段報表.xlsx'
with pd.ExcelWriter(output_path) as writer:
    hour_table.reset_index().to_excel(writer, sheet_name='各小時訂單', index=False)
    detail.to_excel(writer, sheet_name='訂單明細', index=False)
print('重新讀取訂單合計：', pd.read_excel(output_path, sheet_name='各小時訂單')['訂單數'].sum())

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較＋分布】— 各區的送達時間（箱形圖）

> 🎯 **思維重點**：比較 5 個區域的送達分鐘「分布」，用【箱形圖】；資料沒有距離與出餐時間，只能描述差異，不能評斷外送員。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較各區域的「送達分鐘」：
1. 依區域計算訂單數、平均、中位數送達分鐘，依中位數由小到大排序。
2. 畫出箱形圖：X 軸為區域（依中位數排序），Y 軸為送達分鐘。
3. 用文字指出送達時間最長的區域，並說明：資料沒有距離、天氣、出餐時間，不能說該區外送員比較慢。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
area = df.groupby('區域')['送達分鐘'].agg(訂單數='count', 平均='mean', 中位數='median').round(1).sort_values('中位數')
display(area)

plt.figure(figsize=(9, 5))
plt.boxplot([df.loc[df['區域'] == a, '送達分鐘'] for a in area.index], patch_artist=True,
            boxprops=dict(facecolor='#a8dadc', color='#1d3557'), medianprops=dict(color='#e74c3c', linewidth=2))
plt.xticks(range(1, len(area) + 1), area.index)
plt.title('各區域外送送達時間', fontsize=13, fontweight='bold')
plt.xlabel('區域')
plt.ylabel('送達分鐘（分鐘）')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()
print(f"💡 {area.index[-1]} 的送達時間中位數最長。")
print("⚠️ 邊界提醒：資料沒有距離、天氣、出餐時間，不能說該區外送員比較慢。")

### 📝 單元六【選做挑戰】：提問積木【趨勢】— 每日訂單走勢與 7 日移動平均（折線圖）

> 🎯 **思維重點**：每日訂單受週末影響忽高忽低，**7 日移動平均**能看出整體走勢；特殊日子（端午節）要標出來，避免誤判。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用台灣時間的日期分析每日訂單量：
1. 依台灣日期計算每日訂單數，並計算 7 日移動平均。
2. 畫折線圖：細線為每日訂單數、粗線為 7 日移動平均，X 軸為日期，Y 軸從 0 開始，加上圖例。
3. 找出訂單最多的一天並在圖上標註（2025-05-31 為端午節）。
4. 用文字描述兩個月的整體走勢，並提醒：只有兩個月，且節日會拉高移動平均，不能直接推論長期成長。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
daily = df.groupby('日期').size().rename('訂單數').to_frame()
daily.index = pd.to_datetime(daily.index)
daily['7日移動平均'] = daily['訂單數'].rolling(7).mean()
peak_day = daily['訂單數'].idxmax()

plt.figure(figsize=(11, 5))
plt.plot(daily.index, daily['訂單數'], color='#a9cce3', linewidth=1.2, label='每日訂單數')
plt.plot(daily.index, daily['7日移動平均'], color='#1f618d', linewidth=2.5, label='7 日移動平均')
plt.scatter(peak_day, daily.loc[peak_day, '訂單數'], color='#e74c3c', s=80, zorder=3)
plt.annotate(f"{peak_day:%m/%d} 端午節\n{daily.loc[peak_day, '訂單數']} 筆", (peak_day, daily.loc[peak_day, '訂單數']),
             textcoords='offset points', xytext=(-60, -5), color='#e74c3c')
plt.ylim(0, daily['訂單數'].max() * 1.15)
plt.title('每日外送訂單數（台灣時間）', fontsize=13, fontweight='bold')
plt.xlabel('日期')
plt.ylabel('訂單數（筆）')
plt.legend()
plt.grid(linestyle='--', alpha=0.4)
plt.show()
print(f"5 月平均每日 {daily.loc['2025-05', '訂單數'].mean():.1f} 筆；6 月平均每日 {daily.loc['2025-06', '訂單數'].mean():.1f} 筆")
print("⚠️ 邊界提醒：只有兩個月，且端午節會拉高移動平均，不能直接推論長期成長。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：UTC 轉台灣時間後，尖峰為 12 點與 18 點；130 筆訂單日期改變。
- [ ] **報表應用**：Excel 註明台灣時間，訂單合計 2,500 筆。
- [ ] **選做挑戰**：區域比較不推論外送員效率；走勢圖標出端午節並說明期間限制。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。